In [1]:
from tqdm.auto import tqdm
import sympy as sp
from sympy.physics.wigner import clebsch_gordan
from sympy.physics.quantum.spin import Rotation as rot
from sympy import pi
from random import randint

sp.init_printing(use_unicode=True, wrap_line=True, num_columns=120)

import time
start_time = time.time()

Jmax = 3
Lmax = 2

# --- Definizione simboli ---
J, LB, LR, L1, L2, JX, lam, lam1, lam2, eta = sp.symbols('J LB LR L1 L2 JX lam lam1 lam2 eta', integer=True)
theta1, theta2, thetal, thetaK, phi, chi = sp.symbols('theta1 theta2 thetal thetaK phi chi', real=True)
eta_par = sp.symbols('eta_par', integer=True)
Res = sp.symbols('Res')

# --- Helicity amplitudes e altri coefficienti simbolici ---
H = sp.Function('H', complex=True)(eta, J, lam, LB)     # Helicity amplitude per B -> K_J
Hbar = sp.Function('Hbar', complex=True)(eta, J, lam, LB)     # Helicity amplitude per B -> K*_J
LHA = sp.Function('LHA', real=True)(eta_par, lam1, lam2)  # Leptonic Helicity amplitude
G = sp.Function('G', complex=True)(eta, J, lam, LB)  # Helicity Amplitude for K_J -> R P1

# --- Breit-Wigner propagatore ---
Prop = sp.Function('R')(Res)

# --- Blatt-Weisskopf factor ---
L, rBW = sp.symbols('L rBW', positive=True, real=True)
rB, r3, r2 = sp.symbols('rB r3 r2', positive=True, real=True)
q = sp.symbols('q', real=True)
qR = sp.symbols('qR', real=True)


In [22]:
# --- Costruzione dell'ampiezza AR ---
AR = 0
J_val = 1

deltalambda = sp.symbols('deltalambda', integer=True)
lambda1 = sp.symbols('lambda1', real=True)
lambda2 = sp.symbols('lambda2', real=True)

for lam_val in range(-min(1,J_val), min(1,J_val) + 1):
    
    Hterm = H.subs({J: J_val, lam: lam_val, LB: 0})
    term_vector = (
        -Hterm *sp.sqrt(3)/(4*sp.pi)
        * LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
        rot.d(1, lam_val, lambda1 - lambda2, thetal) * rot.d(J_val, lam_val, 0, thetaK) *
        (sp.cos(lam_val*phi) + sp.I*sp.sin(lam_val*phi))
    )
    AR += term_vector

term_timelike = 0
t = sp.symbols('t', real=True)
At = sp.symbols('At', complex=True)
Lt = sp.Function('Lt', complex=True)(lam1, lam2)     # Helicity amplitude per B -> K_J
term_timelike = (
    At * sp.sqrt(3)/(4*sp.pi) *
    Lt.subs(lam1, lambda1).subs(lam2, lambda2) *
    rot.d(J_val, 0, 0, thetaK)
)

AR += term_timelike


# Mostra l'ampiezza totale
print("\n" + "="*50)
if AR == 0:
    print("L'ampiezza AR è nulla (nessuna risonanza selezionata).")
else:
    print("Ampiezza AR totale:")
    print(f"Termini nell'ampiezza AR: {len(AR.as_ordered_terms())}")
print("="*50)

# Mostra l'ampiezza
sp.pprint(AR)


Ampiezza AR totale:
Termini nell'ampiezza AR: 4
                  1                                                                    1             1                 ↪
√3⋅At⋅Lt(λ₁, λ₂)⋅d   (thetaK)   √3⋅(-ⅈ⋅sin(φ) + cos(φ))⋅H(η, 1, -1, 0)⋅LHA(η, λ₁, λ₂)⋅d    (thetaK)⋅d          (thetal ↪
                  0,0                                                                  -1,0          -1,λ₁ - λ₂        ↪
───────────────────────────── - ────────────────────────────────────────────────────────────────────────────────────── ↪
             4⋅π                                                          4⋅π                                          ↪

↪                                                          1            1                                              ↪
↪ )   √3⋅(ⅈ⋅sin(φ) + cos(φ))⋅H(η, 1, 1, 0)⋅LHA(η, λ₁, λ₂)⋅d   (thetaK)⋅d         (thetal)   √3⋅H(η, 1, 0, 0)⋅LHA(η, λ₁ ↪
↪                                                          1,0          1,λ₁ - λ₂                      

In [24]:
b = sp.symbols('b', real = True)
ml = sp.symbols('ml', real = True)

Leptonic_subs = {
    ## Leptonic Helicity Amplitudes (Vettoriali - Standard)
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): (1+b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): (1-b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 2*ml,

    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): (1-b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): (1+b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 2*ml,

    Lt.subs({lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    Lt.subs({lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): -2*ml,
    Lt.subs({lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): 0,
    Lt.subs({lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): 0,
}
# AMPIEZZA TOTALE
Atot = AR
Atotsqfin = 0

for lambda1_val in [sp.Rational(-1,2), sp.Rational(1,2)]:
    for lambda2_val in [sp.Rational(-1,2), sp.Rational(1,2)]:
        
        Term_L = Atot.subs({eta: -1, lambda1: lambda1_val, lambda2: lambda2_val})
        Term_R = Atot.subs({eta: 1, lambda1: lambda1_val, lambda2: lambda2_val})
        
        Amp_Total_Helicity = Term_L + Term_R
        Amp_Total_Helicity = Amp_Total_Helicity.doit().subs(Leptonic_subs)
        
        term_sq = Amp_Total_Helicity * sp.conjugate(Amp_Total_Helicity)
        
        Atotsqfin += term_sq
N = (3*16)/(4*32*sp.pi**2)
Atotsqfin = sp.expand((Atotsqfin/q**2)/N)

terms_list = Atotsqfin.as_ordered_terms()
print(f"Numero di termini: {len(terms_list)}\n")

sp.pprint(Atotsqfin)

Numero di termini: 333

        2    2         __    2                    2       2            2         _______________    2                  ↪
16⋅At⋅ml ⋅cos (thetaK)⋅At   b ⋅H(-1, 1, -1, 0)⋅sin (φ)⋅sin (thetaK)⋅cos (thetal)⋅H(-1, 1, -1, 0)   b ⋅H(-1, 1, -1, 0)⋅ ↪
───────────────────────── + ──────────────────────────────────────────────────────────────────── - ─────────────────── ↪
            2                                                4                                                         ↪
           q                                                                                                           ↪

↪    2       2            2         ______________    2                    2       2            2         ____________ ↪
↪ sin (φ)⋅sin (thetaK)⋅cos (thetal)⋅H(-1, 1, 1, 0)   b ⋅H(-1, 1, -1, 0)⋅sin (φ)⋅sin (thetaK)⋅cos (thetal)⋅H(1, 1, -1,  ↪
↪ ──────────────────────────────────────────────── - ───────────────────────────────────────────────────────────────── ↪
↪      

In [25]:
ortho_basis = {
    # Theta K
    sp.cos(thetaK)**2: (1 + sp.cos(2*thetaK)) / 2,
    sp.sin(thetaK)**2: (1 - sp.cos(2*thetaK)) / 2,
    sp.sin(thetaK)*sp.cos(thetaK): sp.sin(2*thetaK) / 2,
    
    # Theta l
    sp.cos(thetal)**2: (1 + sp.cos(2*thetal)) / 2,
    sp.sin(thetal)**2: (1 - sp.cos(2*thetal)) / 2,
    sp.sin(thetal)*sp.cos(thetal): sp.sin(2*thetal) / 2,
    
    # Phi
    sp.cos(phi)**2: (1 + sp.cos(2*phi)) / 2,
    sp.sin(phi)**2: (1 - sp.cos(2*phi)) / 2,
    sp.sin(phi)*sp.cos(phi): sp.sin(2*phi) / 2,
}

Atotsq_mod = Atotsqfin.subs(ortho_basis).expand()

In [26]:
# Atotsq_mod = Atotsqfin

def extract_trig_structure(expr):
    """Estrae la struttura trigonometrica completa da un'espressione"""
    trig_factors = []
    non_trig_factors = []
    
    # Scompone il termine in fattori
    if expr.is_Mul: #controlla se è un prodotto 'Mul' per Sympy e li ordina come termini
        factors = expr.as_ordered_factors()
    elif expr.is_Pow:
        # Gestisci le potenze
        base, exp = expr.as_base_exp() # Divide in base ed esponente
        if base.atoms(sp.sin, sp.cos):
            return [expr], []  # Tutta la potenza è trigonometrica
        else:
            return [], [expr]  # Tutta la potenza è non-trigonometrica
    else:
        factors = [expr] #in questo caso ci sono più termini di tipo 'Add' sommati tra loro e li mette in una lista
    
    for factor in factors:
        if factor.atoms(sp.sin, sp.cos):
            # Questo fattore contiene funzioni trigonometriche
            if factor.is_Pow:
                # È una potenza di funzione trigonometrica
                trig_factors.append(factor)
            elif factor.is_Mul:
                # È un prodotto che contiene trigonometria
                sub_trig, sub_non_trig = extract_trig_structure(factor)
                trig_factors.extend(sub_trig) #extend aggiunge in coda tutti gli elementi della lista 'sub_trig'
                non_trig_factors.extend(sub_non_trig)
            else:
                # È una singola funzione trigonometrica
                trig_factors.append(factor)
        else:
            non_trig_factors.append(factor)
    
    return trig_factors, non_trig_factors

# Raccoglie tutti i prodotti trigonometrici unici
trig_coefficients = {}

print("Analizzando ogni termine dell'espressione...")
for i, term in tqdm(enumerate(Atotsq_mod.as_ordered_terms()), total=len(Atotsq_mod.as_ordered_terms()), desc="Analisi termini angolari"):
    
    # Estrai la struttura trigonometrica
    trig_factors, non_trig_factors = extract_trig_structure(term)
    
    # Costruisci il prodotto trigonometrico
    if trig_factors:
        trig_product = 1
        for tf in trig_factors:
            trig_product *= tf
        trig_key = str(trig_product)
    else:
        trig_key = "COSTANTE"
    
    # Costruisci il coefficiente non trigonometrico
    if non_trig_factors:
        non_trig_product = 1
        for ntf in non_trig_factors:
            non_trig_product *= ntf
    else:
        non_trig_product = 1
    
    # Accumula i coefficienti
    if trig_key in trig_coefficients:
        trig_coefficients[trig_key] += non_trig_product
    else:
        trig_coefficients[trig_key] = non_trig_product

print(f"\nTrovati {len(trig_coefficients)} prodotti trigonometrici distinti")

# print("Semplificando i coefficienti...")
# for key in tqdm(trig_coefficients, desc="Semplificazione coefficienti", total=len(trig_coefficients)):
#     trig_coefficients[key] = sp.simplify(trig_coefficients[key])


# Ordina per lunghezza della stringa (prima i più semplici)
print("Ordinando i prodotti trigonometrici per complessità")
sorted_keys = sorted(trig_coefficients.keys(), key=lambda x: (len(x), x))

print(f"\n" + "="*70)
print("RIEPILOGO FINALE:")
print("="*70)
print(f"- Prodotti trigonometrici distinti: {len([k for k, v in trig_coefficients.items() if v != 0])}")
for k in range(len(sorted_keys)):
    print(f"  • Prodotto #{k}: {sorted_keys[k]}")

print(f"- Analisi completata con successo!")


Analizzando ogni termine dell'espressione...


Analisi termini angolari:   0%|          | 0/458 [00:00<?, ?it/s]


Trovati 18 prodotti trigonometrici distinti
Ordinando i prodotti trigonometrici per complessità

RIEPILOGO FINALE:
- Prodotti trigonometrici distinti: 18
  • Prodotto #0: COSTANTE
  • Prodotto #1: cos(2*phi)
  • Prodotto #2: sin(2*phi)
  • Prodotto #3: cos(thetal)
  • Prodotto #4: cos(2*thetaK)
  • Prodotto #5: cos(2*thetal)
  • Prodotto #6: cos(2*phi)*cos(2*thetaK)
  • Prodotto #7: cos(2*phi)*cos(2*thetal)
  • Prodotto #8: sin(2*phi)*cos(2*thetaK)
  • Prodotto #9: sin(2*phi)*cos(2*thetal)
  • Prodotto #10: cos(2*thetaK)*cos(thetal)
  • Prodotto #11: cos(2*thetaK)*cos(2*thetal)
  • Prodotto #12: sin(2*thetaK)*sin(thetal)*cos(phi)
  • Prodotto #13: sin(phi)*sin(2*thetaK)*sin(thetal)
  • Prodotto #14: sin(2*thetaK)*sin(2*thetal)*cos(phi)
  • Prodotto #15: sin(phi)*sin(2*thetaK)*sin(2*thetal)
  • Prodotto #16: cos(2*phi)*cos(2*thetaK)*cos(2*thetal)
  • Prodotto #17: sin(2*phi)*cos(2*thetaK)*cos(2*thetal)
- Analisi completata con successo!


In [27]:
A = sp.Function('A', complex=True)(eta, lam) 
t = sp.symbols('t', real=True)
perp = sp.symbols('perp', real=True)
para = sp.symbols('para', real=True)

L = sp.symbols('L', integer=True)
R = sp.symbols('R', integer=True)
helicity_to_transverity_subs = {
    H.subs({eta: -1, J: 1, lam: 0, LB: 0}): A.subs({eta: L, lam: 0}),
    H.subs({eta: -1, J: 1, lam: 1, LB: 0}): A.subs({eta: L, lam: 1}),
    H.subs({eta: -1, J: 1, lam: -1, LB: 0}): A.subs({eta: L, lam: -1}),
    H.subs({eta: 1, J: 1, lam: 0, LB: 0}): A.subs({eta: R, lam: 0}),
    H.subs({eta: 1, J: 1, lam: 1, LB: 0}): A.subs({eta: R, lam: 1}),
    H.subs({eta: 1, J: 1, lam: -1, LB: 0}): A.subs({eta: R, lam: -1})
}

for key in sorted_keys:
    coeff = trig_coefficients[key]
    trig_coefficients[key] = coeff.subs(helicity_to_transverity_subs)

In [28]:
transverity_subs = {
    A.subs({eta: L, lam: 1}): (A.subs({eta:L, lam: para}) + A.subs({eta:L, lam: perp}))/sp.sqrt(2),
    A.subs({eta: L, lam: -1}): (A.subs({eta:L, lam: para}) - A.subs({eta:L, lam: perp}))/sp.sqrt(2),
    A.subs({eta: R, lam: 1}): (A.subs({eta:R, lam: para}) + A.subs({eta:R, lam: perp}))/sp.sqrt(2),
    A.subs({eta: R, lam: -1}): (A.subs({eta:R, lam: para}) - A.subs({eta:R, lam: perp}))/sp.sqrt(2),    
}

for key in sorted_keys:
    coeff = trig_coefficients[key]
    trig_coefficients[key] = sp.expand(coeff.subs(transverity_subs))
    if trig_coefficients[key] == 0:
        print(f"Key:{key} è nullo dopo la sostituzione in transverità.")

In [29]:
simplified_terms = {}

# Dizionario di sostituzioni per pattern A*conjugate(A) -> |A|²
conjugate_subs = {}
for x in [0, t, perp, para]:
    for eta_val in [L, R]:
        conjugate_subs[A.subs({eta: eta_val, lam: x}) * sp.conjugate(A.subs({eta: eta_val, lam: x}))] = sp.Abs(A.subs({eta: eta_val, lam: x}))**2

conjugate_subs[At * sp.conjugate(At)] = sp.Abs(At)**2

for key in tqdm(sorted_keys, desc="Semplificazione con dizionario", total=len(sorted_keys)):
    coeff = trig_coefficients[key]
    
    if coeff == 0:
        simplified_terms[key] = 0
        continue
    
    # Applica le sostituzioni
    simplified = coeff.subs(conjugate_subs)
    simplified_terms[key] = sp.expand(simplified)


Semplificazione con dizionario:   0%|          | 0/18 [00:00<?, ?it/s]

In [30]:
FINAL_trig_coefficients = {}

for key in tqdm(sorted_keys, desc="Semplificazione z±conj(z) con dizionario", total=len(sorted_keys)):
    coeff = simplified_terms[key]
    
    if coeff == 0:
        FINAL_trig_coefficients[key] = 0
        continue
    
    terms = list(coeff.as_ordered_terms())
    
    new_terms = []
    used_indices = set()
    
    for i, term_i in enumerate(terms):
        if i in used_indices:
            continue
        
        found_conjugate = False
        
        # Cerca il coniugato tra gli altri termini
        for j, term_j in enumerate(terms):
            if i == j or j in used_indices:
                continue
            
            # Controlla se term_j == conjugate(term_i) -> SOMMA
            if term_j == sp.conjugate(term_i):
                used_indices.add(i)
                used_indices.add(j)
                found_conjugate = True
                # z + conj(z) = 2*Re(z)
                new_terms.append(2 * sp.re(term_i))
                break
            
            # Controlla se term_j == -conjugate(term_i) -> DIFFERENZA
            elif term_j == -sp.conjugate(term_i):
                used_indices.add(i)
                used_indices.add(j)
                found_conjugate = True
                # z - conj(z) = 2i*Im(z), ma siccome z contiene già I,
                # scriviamo direttamente: (z - conj(z))/i = 2*Im(z) reale
                # Oppure lasciamo: i*(z - conj(z))/2i = (z - conj(z))/2
                # Meglio: z - conj(z) è puramente immaginario, quindi:
                new_terms.append(term_i - sp.conjugate(term_i))
                break
        
        if not found_conjugate:
            # Nessun coniugato trovato, mantieni il termine
            new_terms.append(term_i)
            used_indices.add(i)
    
    # Ricostruisci il coefficiente
    if new_terms:
        FINAL_trig_coefficients[key] = sum(new_terms)

    else:
        FINAL_trig_coefficients[key] = 0



Semplificazione z±conj(z) con dizionario:   0%|          | 0/18 [00:00<?, ?it/s]

In [31]:
###Raccogliamo il possibile nei vari coefficienti finali
collected_FINAL_trig_coefficients = {}

for key in tqdm(sorted_keys, desc="Raccolta termini simili", total=len(sorted_keys)):
    coeff = FINAL_trig_coefficients[key]
    
    if coeff == 0:
        collected_FINAL_trig_coefficients[key] = 0
        continue
    
    # Estrai tutti i moduli quadri, parti reali e immaginarie
    abs_terms = list(coeff.atoms(sp.Abs))
    re_terms = list(coeff.atoms(sp.re))
    im_terms = list(coeff.atoms(sp.im))
    
    # Combina tutti i target di raccolta
    collect_targets = abs_terms + re_terms + im_terms
    
    if collect_targets:
        # Raggruppa rispetto a |A|², Re(...), Im(...)
        collected = sp.collect(coeff, collect_targets)
    else:
        # Se non ci sono target, mantieni così
        collected = coeff
    
    collected_FINAL_trig_coefficients[key] = collected

Raccolta termini simili:   0%|          | 0/18 [00:00<?, ?it/s]

In [32]:
mass={
    ml**2: (q**2/4)*(1 - b**2)
}
beta = {
    b**2: 1 - 4*ml**2/q**2
}

KEY = {}
for i,key in enumerate(sorted_keys):
    KEY[i] = sp.simplify(sorted_keys[i])*collected_FINAL_trig_coefficients[key]

In [33]:
h=17
print(f"{h+1}:{sorted_keys[h]}")
sp.pprint(collected_FINAL_trig_coefficients[sorted_keys[h]].collect(sp.sqrt(2)).subs(mass))

18:sin(2*phi)*cos(2*thetaK)*cos(2*thetal)
   2   ⎛           __________⎞    2   ⎛           __________⎞
  b ⋅im⎝A(L, para)⋅A(L, perp)⎠   b ⋅im⎝A(R, para)⋅A(R, perp)⎠
- ──────────────────────────── - ────────────────────────────
               4                              4              


In [34]:
J1s1c = KEY[0] + KEY[4]
index_key=[0,4]
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")

sp.pprint(J1s1c.subs(mass).subs(ortho_basis).expand().collect([sp.Abs(A.subs({eta:L, lam:0})), sp.Abs(A.subs({eta:L, lam:para})), sp.Abs(A.subs({eta:L, lam:perp}))]).simplify())

0: COSTANTE
4: cos(2*thetaK)
                                              2   ⎛           __________⎞             2   ⎛           __________⎞      ↪
            2   ⎛        _______⎞   COSTANTE⋅b ⋅re⎝A(L, para)⋅A(R, para)⎠   COSTANTE⋅b ⋅re⎝A(L, perp)⋅A(R, perp)⎠      ↪
- COSTANTE⋅b ⋅re⎝A(L, 0)⋅A(R, 0)⎠ - ───────────────────────────────────── - ───────────────────────────────────── - 2⋅ ↪
                                                      2                                       2                        ↪

↪                               2             2             2             2                                            ↪
↪           2     2   COSTANTE⋅b ⋅│A(R, para)│    COSTANTE⋅b ⋅│A(R, perp)│               ⎛        _______⎞   COSTANTE⋅ ↪
↪ COSTANTE⋅b ⋅│At│  + ───────────────────────── + ───────────────────────── + COSTANTE⋅re⎝A(L, 0)⋅A(R, 0)⎠ + ───────── ↪
↪                                 8                           8                                                        ↪

↪

In [35]:
J2s2c = KEY[5] + KEY[11]
index_key=[5,11]
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")

sp.pprint(J2s2c.subs(mass).subs(ortho_basis).expand().collect(sp.Abs(A.subs({eta:L, lam:0}))).simplify())


5: cos(2*thetal)
11: cos(2*thetaK)*cos(2*thetal)
 2 ⎛     2                  2      2                     2      2                     2        2                  2    ↪
b ⋅⎝4⋅sin (thetaK)⋅│A(L, 0)│  + sin (thetaK)⋅│A(L, para)│  + sin (thetaK)⋅│A(L, perp)│  + 4⋅sin (thetaK)⋅│A(R, 0)│  +  ↪
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── ↪
                                                                                                            4          ↪

↪    2                     2      2                     2              2              2⎞              
↪ sin (thetaK)⋅│A(R, para)│  + sin (thetaK)⋅│A(R, perp)│  - 4⋅│A(L, 0)│  - 4⋅│A(R, 0)│ ⎠⋅cos(2⋅thetal)
↪ ────────────────────────────────────────────────────────────────────────────────────────────────────
↪                                                                                                     


In [36]:
J3 = KEY[1] + KEY[6] + KEY[7] + KEY[16]
index_key=[1,6,7,16]
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")
sp.pprint(J3.subs(mass).simplify())

1: cos(2*phi)
6: cos(2*phi)*cos(2*thetaK)
7: cos(2*phi)*cos(2*thetal)
16: cos(2*phi)*cos(2*thetaK)*cos(2*thetal)
  2 ⎛            2               2               2               2⎞    2            2                  
-b ⋅⎝│A(L, para)│  - │A(L, perp)│  + │A(R, para)│  - │A(R, perp)│ ⎠⋅sin (thetaK)⋅sin (thetal)⋅cos(2⋅φ) 
───────────────────────────────────────────────────────────────────────────────────────────────────────
                                                   2                                                   


In [37]:
J4 = -KEY[14]
index_key=[14]
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")
sp.pprint(J4.subs(mass).expand().collect(sp.re(A.subs({eta:L, lam:perp}))).simplify())

14: sin(2*thetaK)*sin(2*thetal)*cos(phi)
    2 ⎛  ⎛        __________⎞     ⎛        __________⎞⎞                                   
√2⋅b ⋅⎝re⎝A(L, 0)⋅A(L, para)⎠ + re⎝A(R, 0)⋅A(R, para)⎠⎠⋅sin(2⋅thetaK)⋅sin(2⋅thetal)⋅cos(φ)
──────────────────────────────────────────────────────────────────────────────────────────
                                            2                                             


In [38]:
J5 = KEY[12]
index_key=[12]
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")
sp.pprint(J5.subs(beta).expand().collect(sp.re(A.subs({eta:L, lam:perp}))).simplify())

12: sin(2*thetaK)*sin(thetal)*cos(phi)
     ⎛  ⎛        __________⎞     ⎛        __________⎞⎞                                 
√2⋅b⋅⎝re⎝A(L, 0)⋅A(L, perp)⎠ - re⎝A(R, 0)⋅A(R, perp)⎠⎠⋅sin(2⋅thetaK)⋅sin(thetal)⋅cos(φ)


In [39]:
J6 = -(KEY[3] + KEY[10])
index_key=[3,10]
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")
sp.pprint(J6.subs(mass).expand().collect(sp.re(A.subs({eta:L, lam:0}))).simplify())

3: cos(thetal)
10: cos(2*thetaK)*cos(thetal)
    ⎛  ⎛           __________⎞     ⎛           __________⎞⎞    2                    
2⋅b⋅⎝re⎝A(L, para)⋅A(L, perp)⎠ - re⎝A(R, para)⋅A(R, perp)⎠⎠⋅sin (thetaK)⋅cos(thetal)


In [40]:
J7 = KEY[13]
index_key=[13]
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")
sp.pprint(J7.subs(mass).expand().collect([sp.im(A.subs({eta:L, lam:perp})*sp.conjugate(A.subs({eta:L, lam:para}))), sp.im(A.subs({eta:R, lam:perp})*sp.conjugate(A.subs({eta:R, lam:para})))]).simplify())

13: sin(phi)*sin(2*thetaK)*sin(thetal)
     ⎛  ⎛        __________⎞     ⎛        __________⎞⎞                                 
√2⋅b⋅⎝im⎝A(L, 0)⋅A(L, para)⎠ - im⎝A(R, 0)⋅A(R, para)⎠⎠⋅sin(φ)⋅sin(2⋅thetaK)⋅sin(thetal)


In [41]:
J8 = -KEY[15]
index_key=[15]
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")
sp.pprint(J8.subs(mass).expand().collect(ml**2).simplify())

15: sin(phi)*sin(2*thetaK)*sin(2*thetal)
    2 ⎛  ⎛        __________⎞     ⎛        __________⎞⎞                                   
√2⋅b ⋅⎝im⎝A(L, 0)⋅A(L, perp)⎠ + im⎝A(R, 0)⋅A(R, perp)⎠⎠⋅sin(φ)⋅sin(2⋅thetaK)⋅sin(2⋅thetal)
──────────────────────────────────────────────────────────────────────────────────────────
                                            2                                             


In [42]:
J9 = -(KEY[2] + KEY[8] + KEY[9] + KEY[17])
index_key=[2,8,9,17]
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")
sp.pprint(J9.subs(mass).expand().simplify())

2: sin(2*phi)
8: sin(2*phi)*cos(2*thetaK)
9: sin(2*phi)*cos(2*thetal)
17: sin(2*phi)*cos(2*thetaK)*cos(2*thetal)
 2 ⎛  ⎛           __________⎞     ⎛           __________⎞⎞             2            2        
b ⋅⎝im⎝A(L, para)⋅A(L, perp)⎠ + im⎝A(R, para)⋅A(R, perp)⎠⎠⋅sin(2⋅φ)⋅sin (thetaK)⋅sin (thetal)
